# 8. Create Source-User Country Networks

This notebook turns the merged LLM-labelled source data into Gephi-ready bipartite networks. The main partition pair is **affiliation/user countries** on one side and **standardized source names** on the other side. Resource nodes retain resource mentions. Holder nodes union the paper-level standardized holder field with holders explicitly inferred through the reviewed resource–holder dictionary, then deduplicate paper–country–holder pairs.


## How To Read This Notebook

The notebook has three main layers:

1. **Full country-source network**: keeps every observed country/source connection.
2. **Simple `weight_gt1` pruning**: keeps only country/source edges that appear in at least two unique papers.
3. **Coscia noise-corrected backbone**: keeps edges that are stronger than expected given each country and source node's total strength.

There is also a **lead-affiliation variant** near the end. That version uses only the first and last listed affiliation countries for each paper, which is a rough proxy for lead/corresponding-country emphasis. It creates its own full and `weight_gt1` files.

In all network tables, `weight` means the number of unique papers connecting the two nodes after deduplicating repeated mentions within the same paper.


## Output Versions Created Here

All files are written to `processed_data`.

**Main full version**: `llm_affiliation_country_database_bipartite_edges.csv`, `llm_affiliation_country_database_bipartite_nodes.csv`, and `llm_affiliation_country_database_bipartite_network.gexf`.

**Main simple-pruned version**: the same prefix plus `_weight_gt1`. This is just the full network with edges of weight 1 removed.

**Main Coscia backbone version**: `llm_affiliation_country_database_bipartite_coscia_noise_corrected_backbone_z1_96_minw2_*`. This uses Michele Coscia's noise-corrected backbone with `z = 1.96`, then keeps only observed weights of at least 2.

**Lead-affiliation full version**: `llm_lead_author_affiliation_country_database_bipartite_*`.

**Lead-affiliation simple-pruned version**: `llm_lead_author_affiliation_country_database_bipartite_*_weight_gt1`.

Older files in `processed_data` with names such as `folded_*` come from previous experiments and are not created by the current cells in this notebook.


## 0. Setup And Output Paths

This cell imports the packages, fixes the project root, names the current merged input file, and defines every output path used later. If you need to know where a network version is saved, start here.


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

from itertools import product
from pathlib import Path
import xml.etree.ElementTree as ET

import pandas as pd

# Local testing root. Later, replace this single value with the envpol share-drive root.
# DATA_ROOT is configured through project_config.
ROOT = DATA_ROOT
PROJECT_ROOT = DATA_ROOT
processed_path = DATA_ROOT / "processed_data"

input_file = processed_path / "llm_labelled_data_sources_merged_bibliometric.csv"
supplementary_file = DATA_ROOT / "interdependence_paper_supplementary_data_new.xlsx"

# Name the paper-level columns used for resource and holder nodes.
SOURCE_CATEGORY_COLUMNS = {
    "data_resource_or_product": "standardized_data_resource_names",
    "data_holder": "standardized_data_holder_names",
}
edge_output_file = processed_path / "llm_affiliation_country_database_bipartite_edges.csv"
node_output_file = processed_path / "llm_affiliation_country_database_bipartite_nodes.csv"
gexf_output_file = processed_path / "llm_affiliation_country_database_bipartite_network.gexf"
pruned_edge_output_file = processed_path / "llm_affiliation_country_database_bipartite_edges_weight_gt1.csv"
pruned_node_output_file = processed_path / "llm_affiliation_country_database_bipartite_nodes_weight_gt1.csv"
pruned_gexf_output_file = processed_path / "llm_affiliation_country_database_bipartite_network_weight_gt1.gexf"

lead_edge_output_file = processed_path / "llm_lead_author_affiliation_country_database_bipartite_edges.csv"
lead_node_output_file = processed_path / "llm_lead_author_affiliation_country_database_bipartite_nodes.csv"
lead_gexf_output_file = processed_path / "llm_lead_author_affiliation_country_database_bipartite_network.gexf"
lead_pruned_edge_output_file = processed_path / "llm_lead_author_affiliation_country_database_bipartite_edges_weight_gt1.csv"
lead_pruned_node_output_file = processed_path / "llm_lead_author_affiliation_country_database_bipartite_nodes_weight_gt1.csv"
lead_pruned_gexf_output_file = processed_path / "llm_lead_author_affiliation_country_database_bipartite_network_weight_gt1.gexf"

print(f"Data root: {DATA_ROOT}")
print(f"Merged source-bibliometric input: {input_file}")


## 1. Helper Functions

These functions do the mechanical work used by all later sections: split semicolon-separated country/source strings, deduplicate repeated values within each paper, build paper-level country-source pairs, and write GEXF XML for Gephi.


In [ ]:
def split_semicolon_values(value):
    if pd.isna(value):
        return []

    items = [item.strip() for item in str(value).split(";")]
    return [
        item
        for item in items
        if item and item.lower() not in {"nan", "<na>", "none"}
    ]


def unique_preserving_order(values):
    return list(dict.fromkeys(values))


def first_last_affiliation_countries(value):
    countries = split_semicolon_values(value)
    if not countries:
        return []

    return unique_preserving_order([countries[0], countries[-1]])



# Yield resource nodes and combined direct-plus-inferred holder nodes.
def iter_source_entities(row):
    used_category_columns = [
        (entity_type, column)
        for entity_type, column in SOURCE_CATEGORY_COLUMNS.items()
        if hasattr(row, column)
    ]

    if used_category_columns:
        # Keep resource nodes at their original standardized resource level.
        resource_column = SOURCE_CATEGORY_COLUMNS["data_resource_or_product"]
        resource_names = unique_preserving_order(
            split_semicolon_values(getattr(row, resource_column, None))
        )
        for resource_name in resource_names:
            yield "data_resource_or_product", resource_name

        # Combine the paper-level holder field with holders inferred from resources.
        holder_column = SOURCE_CATEGORY_COLUMNS["data_holder"]
        paper_field_holders = split_semicolon_values(getattr(row, holder_column, None))
        inferred_holders = [
            holder
            for resource_name in resource_names
            for holder in RESOURCE_TO_HOLDERS.get(resource_name, [])
        ]
        for holder_name in unique_preserving_order(paper_field_holders + inferred_holders):
            yield "data_holder", holder_name
        return

    # Backwards-compatible fallback for older merged files without category columns.
    for source_name in unique_preserving_order(split_semicolon_values(row.databases)):
        yield "uncategorized", source_name



def clean_xml_text(value):
    if pd.isna(value):
        return ""

    text = str(value)
    return "".join(
        ch
        for ch in text
        if (
            ord(ch) in (0x9, 0xA, 0xD)
            or 0x20 <= ord(ch) <= 0xD7FF
            or 0xE000 <= ord(ch) <= 0xFFFD
            or 0x10000 <= ord(ch) <= 0x10FFFF
        )
    )


def build_paper_pairs(frame):
    pair_rows = []
    skipped_rows = []

    for row in frame.itertuples(index=False):
        countries = unique_preserving_order(
            split_semicolon_values(row.affiliations_countries)
        )
        source_entities = list(iter_source_entities(row))

        paper_id = (
            getattr(row, "doi_normalized", None)
            or getattr(row, "doi_bibliometric", None)
            or getattr(row, "file_name_key", None)
            or getattr(row, "file_name", None)
            or getattr(row, "global_index", None)
        )

        if not countries or not source_entities:
            skipped_rows.append(
                {
                    "paper_id": paper_id,
                    "global_index": getattr(row, "global_index", None),
                    "file_name": getattr(row, "file_name", None),
                    "doi_normalized": getattr(row, "doi_normalized", None),
                    "title": getattr(row, "title", None),
                    "countries_found": len(countries),
                    "databases_found": len(source_entities),
                }
            )
            continue

        for country, (entity_type, database) in product(countries, source_entities):
            pair_rows.append(
                {
                    "paper_id": paper_id,
                    "global_index": getattr(row, "global_index", None),
                    "file_name": getattr(row, "file_name", None),
                    "doi_bibliometric": getattr(row, "doi_bibliometric", None),
                    "country": country,
                    "database": database,
                    "entity_type": entity_type,
                    "title": getattr(row, "title", None),
                }
            )

    pair_frame = pd.DataFrame(
        pair_rows,
        columns=[
            "paper_id",
            "global_index",
            "file_name",
            "doi_bibliometric",
            "country",
            "database",
            "entity_type",
            "title",
        ],
    )
    if not pair_frame.empty:
        pair_frame = pair_frame.drop_duplicates(
            subset=["paper_id", "country", "database", "entity_type"]
        ).reset_index(drop=True)

    skipped_frame = pd.DataFrame(
        skipped_rows,
        columns=[
            "paper_id",
            "global_index",
            "file_name",
            "doi_normalized",
            "title",
            "countries_found",
            "databases_found",
        ],
    )
    return pair_frame, skipped_frame


def indent_xml(element, level=0):
    indentation = "\n" + "  " * level
    child_indentation = "\n" + "  " * (level + 1)
    children = list(element)

    if children:
        if not element.text or not element.text.strip():
            element.text = child_indentation
        for child in children:
            indent_xml(child, level + 1)
        if not children[-1].tail or not children[-1].tail.strip():
            children[-1].tail = indentation

    if level and (not element.tail or not element.tail.strip()):
        element.tail = indentation


def write_gexf(nodes_df, edges_df, output_path):
    gexf = ET.Element(
        "gexf",
        {
            "xmlns": "http://www.gexf.net/1.2draft",
            "xmlns:xsi": "http://www.w3.org/2001/XMLSchema-instance",
            "xsi:schemaLocation": (
                "http://www.gexf.net/1.2draft "
                "http://www.gexf.net/1.2draft/gexf.xsd"
            ),
            "version": "1.2",
        },
    )

    meta = ET.SubElement(
        gexf,
        "meta",
        {"lastmodifieddate": pd.Timestamp.today().strftime("%Y-%m-%d")},
    )
    ET.SubElement(meta, "creator").text = (
        "Create affiliation country-database bipartite network.ipynb"
    )
    ET.SubElement(meta, "description").text = (
        "Bipartite network linking affiliation countries to standardized LLM-labelled data-source names."
    )

    graph = ET.SubElement(
        gexf,
        "graph",
        {"mode": "static", "defaultedgetype": "undirected"},
    )

    attributes = ET.SubElement(
        graph,
        "attributes",
        {"class": "node", "mode": "static"},
    )
    ET.SubElement(attributes, "attribute", {"id": "0", "title": "node_type", "type": "string"})
    ET.SubElement(attributes, "attribute", {"id": "1", "title": "paper_count", "type": "integer"})
    ET.SubElement(attributes, "attribute", {"id": "2", "title": "entity_type", "type": "string"})

    edge_attributes = ET.SubElement(
        graph,
        "attributes",
        {"class": "edge", "mode": "static"},
    )
    ET.SubElement(edge_attributes, "attribute", {"id": "3", "title": "entity_type", "type": "string"})

    nodes_element = ET.SubElement(graph, "nodes")
    for row in nodes_df.itertuples(index=False):
        node_element = ET.SubElement(
            nodes_element,
            "node",
            {"id": clean_xml_text(row.node_id), "label": clean_xml_text(row.label)},
        )
        attvalues = ET.SubElement(node_element, "attvalues")
        ET.SubElement(
            attvalues,
            "attvalue",
            {"for": "0", "value": clean_xml_text(row.node_type)},
        )
        ET.SubElement(
            attvalues,
            "attvalue",
            {"for": "1", "value": clean_xml_text(int(row.paper_count))},
        )
        ET.SubElement(
            attvalues,
            "attvalue",
            {"for": "2", "value": clean_xml_text(getattr(row, "entity_type", ""))},
        )

    edges_element = ET.SubElement(graph, "edges")
    for edge_id, row in enumerate(edges_df.itertuples(index=False)):
        edge_element = ET.SubElement(
            edges_element,
            "edge",
            {
                "id": str(edge_id),
                "source": clean_xml_text(row.source),
                "target": clean_xml_text(row.target),
                "weight": clean_xml_text(row.weight),
            },
        )
        attvalues = ET.SubElement(edge_element, "attvalues")
        ET.SubElement(
            attvalues,
            "attvalue",
            {"for": "3", "value": clean_xml_text(getattr(row, "entity_type", ""))},
        )

    indent_xml(gexf)
    tree = ET.ElementTree(gexf)
    tree.write(output_path, encoding="utf-8", xml_declaration=True)


## 2. Load Current Merged Data

This cell reads `llm_labelled_data_sources_merged_bibliometric.csv`, keeps matched rows with affiliation countries and source names, and expands each paper into country-source pairs. The result, `paper_pairs`, is the basic long table used by the rest of the notebook.


In [ ]:
# Load the merged paper data.
source_df = pd.read_csv(input_file, low_memory=False)

# Load and split the reviewed resource-to-holder dictionary.
resource_holder_rows = pd.read_excel(
    supplementary_file,
    sheet_name="standardized_sources",
    usecols=["standardized_data_resource_name", "standardized_data_holder_name"],
)
resource_holder_rows["standardized_data_holder_name"] = resource_holder_rows[
    "standardized_data_holder_name"
].map(split_semicolon_values)
resource_holder_rows = (
    resource_holder_rows.explode("standardized_data_holder_name")
    .dropna(subset=["standardized_data_resource_name", "standardized_data_holder_name"])
    .drop_duplicates()
)
RESOURCE_TO_HOLDERS = (
    resource_holder_rows.groupby("standardized_data_resource_name")[
        "standardized_data_holder_name"
    ]
    .agg(lambda values: unique_preserving_order(list(values)))
    .to_dict()
)

# Keep matched rows when that status is available.
analysis_df = source_df.copy()
if "merge_status" in analysis_df.columns:
    analysis_df = analysis_df[analysis_df["merge_status"].eq("matched")].copy()

# Keep papers with affiliation countries and at least one categorized source.
available_source_columns = [
    column for column in SOURCE_CATEGORY_COLUMNS.values() if column in analysis_df.columns
]
if available_source_columns:
    has_source = analysis_df[available_source_columns].notna().any(axis=1)
else:
    has_source = analysis_df["databases"].notna()
analysis_df = analysis_df[
    analysis_df["affiliations_countries"].notna() & has_source
].copy()

# Build deduplicated paper-country-source pairs for the network.
paper_pairs, skipped_rows = build_paper_pairs(analysis_df)

print(f"Loaded {len(source_df)} rows from {input_file.name}.")
print(f"Rows kept for network construction: {len(analysis_df)}")
print(f"Unique papers in network: {paper_pairs['paper_id'].nunique() if not paper_pairs.empty else 0}")
print(f"Paper-country-database pairs: {len(paper_pairs)}")
print(f"Skipped rows after parsing: {len(skipped_rows)}")

paper_pairs.head()


## 3. Build The Main Full Network Tables

This cell aggregates `paper_pairs` into weighted edges and creates the corresponding node table. Countries become `affiliation_country` nodes. Standardized source names become `source_name` nodes, with `entity_type` preserving whether the source is a data resource/product or a data holder.


In [ ]:
edge_table = (
    paper_pairs.groupby(["country", "database", "entity_type"], as_index=False)
    .agg(weight=("paper_id", "nunique"))
    .sort_values(["weight", "country", "entity_type", "database"], ascending=[False, True, True, True])
    .reset_index(drop=True)
)

country_nodes = (
    paper_pairs.groupby("country", as_index=False)
    .agg(paper_count=("paper_id", "nunique"))
    .rename(columns={"country": "label"})
    .assign(node_type="affiliation_country", entity_type="affiliation_country")
)

source_nodes = (
    paper_pairs.groupby(["entity_type", "database"], as_index=False)
    .agg(paper_count=("paper_id", "nunique"))
    .rename(columns={"database": "label"})
    .assign(node_type="source_name")
)

node_table = (
    pd.concat([country_nodes, source_nodes], ignore_index=True)
    .sort_values(["node_type", "entity_type", "label"])
    .reset_index(drop=True)
)
node_table.insert(0, "node_id", range(len(node_table)))

node_lookup = node_table.set_index(["node_type", "entity_type", "label"])["node_id"].to_dict()

edge_table = edge_table.assign(
    source=edge_table["country"].map(
        lambda label: node_lookup[("affiliation_country", "affiliation_country", label)]
    ),
    target=edge_table.apply(
        lambda row: node_lookup[("source_name", row["entity_type"], row["database"])],
        axis=1,
    ),
    source_type="affiliation_country",
    target_type="source_name",
)

edge_table = edge_table[
    ["source", "target", "source_type", "target_type", "country", "database", "entity_type", "weight"]
]

summary = pd.DataFrame(
    {
        "metric": [
            "papers",
            "country_nodes",
            "source_nodes",
            "total_nodes",
            "edges",
        ],
        "value": [
            paper_pairs["paper_id"].nunique() if not paper_pairs.empty else 0,
            len(country_nodes),
            len(source_nodes),
            len(node_table),
            len(edge_table),
        ],
    }
)

edge_entity_type_summary = (
    edge_table.groupby("entity_type", as_index=False)
    .agg(edges=("database", "size"), total_weight=("weight", "sum"))
    .sort_values(["total_weight", "entity_type"], ascending=[False, True])
)

print(summary.to_string(index=False))
print("\nEdge summary by entity type:")
print(edge_entity_type_summary.to_string(index=False))
edge_table.head(20)


## 4. Save The Main Full Version

This cell writes the complete network: every country-source edge with weight at least 1. Use this when you need the full record, including one-off connections.


In [ ]:
edge_table.to_csv(edge_output_file, index=False)
node_table.to_csv(node_output_file, index=False)
write_gexf(node_table, edge_table, gexf_output_file)

print(f"Saved edge table to: {edge_output_file}")
print(f"Saved node table to: {node_output_file}")
print(f"Saved Gephi network to: {gexf_output_file}")


## 5. Save The Simple `weight_gt1` Version

This section removes all edges with `weight == 1`, then drops any nodes that no longer have an edge. It is a quick readability filter, not a statistical backbone. Use this version when the full network is too noisy but you still want a transparent rule.


In [ ]:
pruned_edge_table = edge_table.loc[edge_table["weight"].gt(1)].copy()
referenced_node_ids = set(pruned_edge_table["source"]).union(set(pruned_edge_table["target"]))
pruned_node_table = node_table.loc[node_table["node_id"].isin(referenced_node_ids)].copy()

missing_node_ids = referenced_node_ids - set(pruned_node_table["node_id"])
if missing_node_ids:
    raise ValueError(f"Missing node ids in pruned node table: {sorted(missing_node_ids)[:10]}")

pruned_edge_table.to_csv(pruned_edge_output_file, index=False)
pruned_node_table.to_csv(pruned_node_output_file, index=False)
write_gexf(pruned_node_table, pruned_edge_table, pruned_gexf_output_file)

print(f"Pruned edges with weight > 1: {len(pruned_edge_table):,}")
print(f"Pruned nodes: {len(pruned_node_table):,}")
print(f"Saved pruned edge table to: {pruned_edge_output_file}")
print(f"Saved pruned node table to: {pruned_node_output_file}")
print(f"Saved pruned Gephi network to: {pruned_gexf_output_file}")


## 6. Save The Coscia Noise-Corrected Backbone Version

The full country-source network is dense, so this section applies Michele Coscia's `noise_corrected` backbone. The algorithm scores whether each edge is stronger than expected from the total strength of its country node and source node. This notebook uses `z = 1.96` and then applies `min_weight = 2`.

This creates the `llm_affiliation_country_database_bipartite_coscia_noise_corrected_backbone_z1_96_minw2_*` files.


In [ ]:
import sys

COSCIA_BACKBONING_PATH = PROJECT_ROOT / "code" / "vendor" / "michele_coscia_backboning"
if not (COSCIA_BACKBONING_PATH / "backboning.py").exists():
    raise FileNotFoundError(
        f"Michele Coscia backboning.py was not found at {COSCIA_BACKBONING_PATH}. "
        "Download the Python 3 version from https://www.michelecoscia.com/?page_id=287 "
        "and place backboning.py in that folder."
    )

if str(COSCIA_BACKBONING_PATH) not in sys.path:
    sys.path.insert(0, str(COSCIA_BACKBONING_PATH))

try:
    import backboning as coscia_backboning
except ModuleNotFoundError as exc:
    missing_package = exc.name or "a dependency"
    raise ModuleNotFoundError(
        "Michele Coscia's backboning.py requires scipy and networkx. "
        f"Missing package: {missing_package}. Install them in this notebook environment with "
        "`pip install scipy networkx`, then rerun this section."
    ) from exc

COSCIA_NOISE_CORRECTED_THRESHOLD = 1.96
COSCIA_BACKBONE_MIN_EDGE_WEIGHT = 2
COSCIA_BACKBONE_METHOD = "noise_corrected"
COSCIA_BACKBONE_THRESHOLD_LABEL = str(COSCIA_NOISE_CORRECTED_THRESHOLD).replace(".", "_")

coscia_backbone_prefix = (
    "llm_affiliation_country_database_bipartite_"
    f"coscia_{COSCIA_BACKBONE_METHOD}_backbone_"
    f"z{COSCIA_BACKBONE_THRESHOLD_LABEL}_"
    f"minw{COSCIA_BACKBONE_MIN_EDGE_WEIGHT}"
)
coscia_backbone_edge_output_file = processed_path / f"{coscia_backbone_prefix}_edges.csv"
coscia_backbone_node_output_file = processed_path / f"{coscia_backbone_prefix}_nodes.csv"
coscia_backbone_gexf_output_file = processed_path / f"{coscia_backbone_prefix}_network.gexf"


def _gexf_attribute_type(series):
    if pd.api.types.is_integer_dtype(series):
        return "integer"
    if pd.api.types.is_float_dtype(series):
        return "double"
    return "string"


def write_bipartite_coscia_backbone_gexf(nodes_df, edges_df, output_path):
    gexf = ET.Element(
        "gexf",
        {
            "xmlns": "http://www.gexf.net/1.2draft",
            "xmlns:xsi": "http://www.w3.org/2001/XMLSchema-instance",
            "xsi:schemaLocation": (
                "http://www.gexf.net/1.2draft "
                "http://www.gexf.net/1.2draft/gexf.xsd"
            ),
            "version": "1.2",
        },
    )
    meta = ET.SubElement(
        gexf,
        "meta",
        {"lastmodifieddate": pd.Timestamp.today().strftime("%Y-%m-%d")},
    )
    ET.SubElement(meta, "creator").text = (
        "Create affiliation country-database bipartite network.ipynb"
    )
    ET.SubElement(meta, "description").text = (
        "Michele Coscia noise-corrected backbone of the main affiliation-country "
        "to data-source bipartite network."
    )

    graph = ET.SubElement(gexf, "graph", {"mode": "static", "defaultedgetype": "undirected"})

    node_attribute_columns = ["node_type", "paper_count", "entity_type"]
    edge_attribute_columns = [
        "source_type",
        "target_type",
        "country",
        "database",
        "entity_type",
        "coscia_nij",
        "coscia_nc_score",
        "coscia_nc_sdev",
        "coscia_nc_z_margin",
        "coscia_nc_threshold",
        "coscia_backbone_min_edge_weight",
        "coscia_backbone_method",
    ]
    edge_attribute_columns = [column for column in edge_attribute_columns if column in edges_df.columns]

    node_attributes = ET.SubElement(graph, "attributes", {"class": "node", "mode": "static"})
    node_attribute_ids = {}
    for idx, column in enumerate(node_attribute_columns):
        attr_id = str(idx)
        node_attribute_ids[column] = attr_id
        ET.SubElement(
            node_attributes,
            "attribute",
            {"id": attr_id, "title": column, "type": _gexf_attribute_type(nodes_df[column])},
        )

    edge_attributes = ET.SubElement(graph, "attributes", {"class": "edge", "mode": "static"})
    edge_attribute_ids = {}
    for idx, column in enumerate(edge_attribute_columns, start=len(node_attribute_columns)):
        attr_id = str(idx)
        edge_attribute_ids[column] = attr_id
        ET.SubElement(
            edge_attributes,
            "attribute",
            {"id": attr_id, "title": column, "type": _gexf_attribute_type(edges_df[column])},
        )

    nodes_element = ET.SubElement(graph, "nodes")
    for row in nodes_df.itertuples(index=False):
        node_element = ET.SubElement(
            nodes_element,
            "node",
            {"id": clean_xml_text(row.node_id), "label": clean_xml_text(row.label)},
        )
        attvalues = ET.SubElement(node_element, "attvalues")
        for column, attr_id in node_attribute_ids.items():
            ET.SubElement(
                attvalues,
                "attvalue",
                {"for": attr_id, "value": clean_xml_text(getattr(row, column))},
            )

    edges_element = ET.SubElement(graph, "edges")
    for edge_id, row in enumerate(edges_df.itertuples(index=False)):
        edge_element = ET.SubElement(
            edges_element,
            "edge",
            {
                "id": str(edge_id),
                "source": clean_xml_text(row.source),
                "target": clean_xml_text(row.target),
                "weight": clean_xml_text(row.weight),
            },
        )
        attvalues = ET.SubElement(edge_element, "attvalues")
        for column, attr_id in edge_attribute_ids.items():
            ET.SubElement(
                attvalues,
                "attvalue",
                {"for": attr_id, "value": clean_xml_text(getattr(row, column))},
            )

    indent_xml(gexf)
    ET.ElementTree(gexf).write(output_path, encoding="utf-8", xml_declaration=True)


coscia_input = edge_table[["source", "target", "weight"]].rename(
    columns={"source": "src", "target": "trg", "weight": "nij"}
).copy()
coscia_input["nij"] = pd.to_numeric(coscia_input["nij"], errors="coerce").fillna(0).astype(float)
coscia_input = coscia_input.loc[coscia_input["nij"].gt(0)].copy()

coscia_scored_edges = coscia_backboning.noise_corrected(
    coscia_input,
    undirected=False,
    return_self_loops=False,
    calculate_p_value=False,
)
coscia_thresholded_edges = coscia_backboning.thresholding(
    coscia_scored_edges,
    COSCIA_NOISE_CORRECTED_THRESHOLD,
)

coscia_scores = coscia_scored_edges.rename(
    columns={
        "src": "source",
        "trg": "target",
        "nij": "coscia_nij",
        "score": "coscia_nc_score",
        "sdev_cij": "coscia_nc_sdev",
    }
).copy()
coscia_scores["source"] = coscia_scores["source"].astype(edge_table["source"].dtype)
coscia_scores["target"] = coscia_scores["target"].astype(edge_table["target"].dtype)
coscia_scores["coscia_nc_z_margin"] = (
    coscia_scores["coscia_nc_score"]
    - (COSCIA_NOISE_CORRECTED_THRESHOLD * coscia_scores["coscia_nc_sdev"])
)
coscia_scores["coscia_nc_threshold"] = COSCIA_NOISE_CORRECTED_THRESHOLD
coscia_scores["coscia_backbone_min_edge_weight"] = COSCIA_BACKBONE_MIN_EDGE_WEIGHT
coscia_scores["coscia_backbone_method"] = COSCIA_BACKBONE_METHOD

coscia_thresholded_pairs = coscia_thresholded_edges.rename(
    columns={"src": "source", "trg": "target"}
)[["source", "target"]].drop_duplicates()
coscia_thresholded_pairs["source"] = coscia_thresholded_pairs["source"].astype(edge_table["source"].dtype)
coscia_thresholded_pairs["target"] = coscia_thresholded_pairs["target"].astype(edge_table["target"].dtype)

coscia_backbone_edge_table = (
    edge_table.merge(coscia_thresholded_pairs, on=["source", "target"], how="inner")
    .merge(
        coscia_scores[
            [
                "source",
                "target",
                "coscia_nij",
                "coscia_nc_score",
                "coscia_nc_sdev",
                "coscia_nc_z_margin",
                "coscia_nc_threshold",
                "coscia_backbone_min_edge_weight",
                "coscia_backbone_method",
            ]
        ],
        on=["source", "target"],
        how="left",
    )
    .sort_values(
        ["coscia_nc_z_margin", "weight", "country", "entity_type", "database"],
        ascending=[False, False, True, True, True],
    )
    .loc[lambda frame: frame["weight"].ge(COSCIA_BACKBONE_MIN_EDGE_WEIGHT)]
    .reset_index(drop=True)
)
coscia_backbone_edge_table["coscia_nij"] = coscia_backbone_edge_table["coscia_nij"].astype(int)

coscia_backbone_node_ids = set(coscia_backbone_edge_table["source"]).union(
    set(coscia_backbone_edge_table["target"])
)
coscia_backbone_node_table = node_table.loc[
    node_table["node_id"].isin(coscia_backbone_node_ids)
].copy()

coscia_backbone_edge_table.to_csv(coscia_backbone_edge_output_file, index=False)
coscia_backbone_node_table.to_csv(coscia_backbone_node_output_file, index=False)
write_bipartite_coscia_backbone_gexf(
    coscia_backbone_node_table,
    coscia_backbone_edge_table,
    coscia_backbone_gexf_output_file,
)

print(
    f"Coscia noise-corrected backbone kept {len(coscia_backbone_edge_table):,} of "
    f"{len(edge_table):,} edges ({len(coscia_backbone_edge_table) / len(edge_table):.1%}) "
    f"using z={COSCIA_NOISE_CORRECTED_THRESHOLD} and "
    f"min_weight={COSCIA_BACKBONE_MIN_EDGE_WEIGHT}."
)
print(f"Backbone nodes: {len(coscia_backbone_node_table):,}")
print(f"Saved Coscia backbone edge table to: {coscia_backbone_edge_output_file}")
print(f"Saved Coscia backbone node table to: {coscia_backbone_node_output_file}")
print(f"Saved Coscia backbone Gephi network to: {coscia_backbone_gexf_output_file}")

coscia_backbone_edge_table.head(20)


## 7. Build The Lead-Affiliation Variant

This final network repeats the main full and `weight_gt1` logic, but first replaces each paper's full affiliation-country list with only the first and last country in that list. The goal is to approximate a lead/corresponding-country perspective without changing the source-name side of the network.

This creates the `llm_lead_author_affiliation_country_database_bipartite_*` full files and the matching `_weight_gt1` files. It does **not** create a Coscia backbone for the lead-affiliation variant.


In [ ]:
lead_analysis_df = analysis_df.copy()
lead_analysis_df["affiliations_countries"] = lead_analysis_df[
    "affiliations_countries"
].map(lambda value: "; ".join(first_last_affiliation_countries(value)))

lead_paper_pairs, lead_skipped_rows = build_paper_pairs(lead_analysis_df)

lead_edge_table = (
    lead_paper_pairs.groupby(["country", "database", "entity_type"], as_index=False)
    .agg(weight=("paper_id", "nunique"))
    .sort_values(["weight", "country", "entity_type", "database"], ascending=[False, True, True, True])
    .reset_index(drop=True)
)

lead_country_nodes = (
    lead_paper_pairs.groupby("country", as_index=False)
    .agg(paper_count=("paper_id", "nunique"))
    .rename(columns={"country": "label"})
    .assign(node_type="affiliation_country", entity_type="affiliation_country")
)

lead_source_nodes = (
    lead_paper_pairs.groupby(["entity_type", "database"], as_index=False)
    .agg(paper_count=("paper_id", "nunique"))
    .rename(columns={"database": "label"})
    .assign(node_type="source_name")
)

lead_node_table = (
    pd.concat([lead_country_nodes, lead_source_nodes], ignore_index=True)
    .sort_values(["node_type", "entity_type", "label"])
    .reset_index(drop=True)
)
lead_node_table.insert(0, "node_id", range(len(lead_node_table)))

lead_node_lookup = lead_node_table.set_index(["node_type", "entity_type", "label"])["node_id"].to_dict()

lead_edge_table = lead_edge_table.assign(
    source=lead_edge_table["country"].map(
        lambda label: lead_node_lookup[("affiliation_country", "affiliation_country", label)]
    ),
    target=lead_edge_table.apply(
        lambda row: lead_node_lookup[("source_name", row["entity_type"], row["database"])],
        axis=1,
    ),
    source_type="affiliation_country",
    target_type="source_name",
)

lead_edge_table = lead_edge_table[
    ["source", "target", "source_type", "target_type", "country", "database", "entity_type", "weight"]
]

lead_summary = pd.DataFrame(
    {
        "metric": [
            "papers",
            "country_nodes",
            "source_nodes",
            "total_nodes",
            "edges",
        ],
        "value": [
            lead_paper_pairs["paper_id"].nunique() if not lead_paper_pairs.empty else 0,
            len(lead_country_nodes),
            len(lead_source_nodes),
            len(lead_node_table),
            len(lead_edge_table),
        ],
    }
)

lead_edge_entity_type_summary = (
    lead_edge_table.groupby("entity_type", as_index=False)
    .agg(edges=("database", "size"), total_weight=("weight", "sum"))
    .sort_values(["total_weight", "entity_type"], ascending=[False, True])
)

lead_edge_table.to_csv(lead_edge_output_file, index=False)
lead_node_table.to_csv(lead_node_output_file, index=False)
write_gexf(lead_node_table, lead_edge_table, lead_gexf_output_file)

lead_pruned_edge_table = lead_edge_table.loc[lead_edge_table["weight"].gt(1)].copy()
lead_referenced_node_ids = set(lead_pruned_edge_table["source"]).union(set(lead_pruned_edge_table["target"]))
lead_pruned_node_table = lead_node_table.loc[lead_node_table["node_id"].isin(lead_referenced_node_ids)].copy()

lead_missing_node_ids = lead_referenced_node_ids - set(lead_pruned_node_table["node_id"])
if lead_missing_node_ids:
    raise ValueError(f"Missing node ids in lead pruned node table: {sorted(lead_missing_node_ids)[:10]}")

lead_pruned_edge_table.to_csv(lead_pruned_edge_output_file, index=False)
lead_pruned_node_table.to_csv(lead_pruned_node_output_file, index=False)
write_gexf(lead_pruned_node_table, lead_pruned_edge_table, lead_pruned_gexf_output_file)

print(f"Lead-affiliation paper-country-database pairs: {len(lead_paper_pairs):,}")
print(f"Skipped lead-affiliation rows after parsing: {len(lead_skipped_rows):,}")
print(lead_summary.to_string(index=False))
print("\nLead-affiliation edge summary by entity type:")
print(lead_edge_entity_type_summary.to_string(index=False))
print(f"Saved lead edge table to: {lead_edge_output_file}")
print(f"Saved lead node table to: {lead_node_output_file}")
print(f"Saved lead Gephi network to: {lead_gexf_output_file}")
print(f"Saved lead pruned edge table to: {lead_pruned_edge_output_file}")
print(f"Saved lead pruned node table to: {lead_pruned_node_output_file}")
print(f"Saved lead pruned Gephi network to: {lead_pruned_gexf_output_file}")
